# 01 — NLLB: zero-shot, target-only, rehearsal
Uses Meta’s **lid218e language identifier**, not an NLLB translation transformer. Loads the full fastText checkpoint and continues its original softmax classifier.

Run notebook 00 first. Keep the same data and settings for every model. Checkpoints are large; run one model at a time.

Table 2 measures whether forgetting occurs. Table 3 measures mitigation. Neither outcome is assumed.

In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import os, sys
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/lid_finetuning_bundle')]
ROOT = next((p for p in candidates if (p/'config.json').exists() and (p/'lidlab').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Extract the complete ZIP first and set ROOT to its lid_finetuning_bundle folder.')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print('Project folder:', ROOT)


Project folder: d:\Projects\ML Projects\LangID - DSE project\data_pipeline\new_method


In [2]:
from lidlab.data import load_config
from lidlab.experiment import run_experiment
c = load_config()
MODEL = 'nllb'
print('Starting checkpoint:', c['models'][MODEL])
print('Replay starts from:', c['replay_start'])

Starting checkpoint: {'repo_id': 'facebook/fasttext-language-identification', 'filename': 'model.bin', 'revision': None, 'local_path': None, 'lr': 0.05, 'existing_zero_predictions': None}
Replay starts from: base


## Run all three conditions
The first run downloads and locks the original checkpoint. Both trained conditions keep every original label and add missing target labels. An extra untrained `initialized` evaluation measures the effect of adding labels alone.

Existing zero-shot predictions are imported only if configured and aligned with the evaluation index. Otherwise zero-shot is recalculated. Completed unchanged stages are reused. Interrupted stages restart from their defined initial checkpoint.

If you change data or hyperparameters, use a new `output_dir` before rerunning.

In [3]:
results = run_experiment(c, MODEL)
for phase, benchmark_results in results.items():
    for benchmark, frame in benchmark_results.items():
        print(phase, benchmark)
        display(frame[['language','precision','recall','f1','support']])

nllb zero_shot {'commonlid': 0.7443, 'flores_plus': 0.743, 'wili_2018': 0.7614}
nllb initialized {'commonlid': 0.7443, 'flores_plus': 0.743, 'wili_2018': 0.7614}
nllb target_only {'commonlid': 0.9581, 'flores_plus': 0.967, 'wili_2018': 0.9845}
nllb replay {'commonlid': 0.9693, 'flores_plus': 0.9644, 'wili_2018': 0.9798}
zero_shot commonlid


,language,precision,recall,f1,support
0,sin_Sinh,0.377096,0.976977,0.544157,2693
1,pli_Sinh,0.000000,0.000000,0.000000,3027
2,san_Sinh,0.000000,0.000000,0.000000,1333
3,san_Deva,0.981320,0.886389,0.931442,889
4,eng_Latn,0.988645,0.866159,0.923358,27443
5,tam_Taml,1.000000,0.950617,0.974684,81
6,hin_Deva,0.990786,0.938881,0.964136,3665
7,ben_Beng,1.000000,0.973461,0.986552,1884
8,arb_Arab,0.999538,0.986766,0.993111,41634
9,fra_Latn,0.950421,0.908050,0.928752,3230


zero_shot flores_plus


,language,precision,recall,f1,support
0,sin_Sinh,0.377096,0.976977,0.544157,2693
1,pli_Sinh,0.000000,0.000000,0.000000,3027
2,san_Sinh,0.000000,0.000000,0.000000,1327
3,san_Deva,1.000000,0.978261,0.989011,1012
4,eng_Latn,0.968421,1.000000,0.983957,1012
5,tam_Taml,1.000000,1.000000,1.000000,1012
6,hin_Deva,0.983447,0.998024,0.990682,1012
7,ben_Beng,1.000000,1.000000,1.000000,1012
8,arb_Arab,0.999013,0.500000,0.666447,2024
9,fra_Latn,1.000000,0.999012,0.999506,1012


zero_shot wili_2018


...

,language,precision,recall,f1,support
0,sin_Sinh,0.993953,0.976606,0.985203,2693
1,pli_Sinh,0.987853,0.994054,0.990944,3027
2,san_Sinh,0.995472,0.989497,0.992476,1333
3,san_Deva,0.895119,0.969629,0.930886,889
4,eng_Latn,0.977615,0.919834,0.947845,27443
5,tam_Taml,1.000000,0.975309,0.987500,81
6,hin_Deva,0.997648,0.926057,0.960521,3665
7,ben_Beng,1.000000,0.971868,0.985734,1884
8,arb_Arab,0.999927,0.985853,0.992840,41634
9,fra_Latn,0.955349,0.920743,0.937727,3230


replay flores_plus


,language,precision,recall,f1,support
0,sin_Sinh,0.993953,0.976606,0.985203,2693
1,pli_Sinh,0.998672,0.994054,0.996358,3027
2,san_Sinh,0.995472,0.993971,0.994721,1327
3,san_Deva,1.000000,1.000000,1.000000,1012
4,eng_Latn,0.934441,1.000000,0.966110,1012
5,tam_Taml,1.000000,1.000000,1.000000,1012
6,hin_Deva,1.000000,0.999012,0.999506,1012
7,ben_Beng,1.000000,1.000000,1.000000,1012
8,arb_Arab,1.000000,0.500000,0.666667,2024
9,fra_Latn,1.000000,1.000000,1.000000,1012


replay wili_2018


,language,precision,recall,f1,support
0,sin_Sinh,0.993953,0.976606,0.985203,2693
1,pli_Sinh,0.998672,0.994054,0.996358,3027
2,san_Sinh,0.995472,0.984328,0.989869,1340
3,san_Deva,0.996954,0.998983,0.997967,983
4,eng_Latn,0.830833,0.997000,0.906364,1000
5,tam_Taml,1.000000,0.988810,0.994373,983
6,hin_Deva,1.000000,0.977978,0.988866,999
7,ben_Beng,1.000000,0.891000,0.942359,1000
8,arb_Arab,1.000000,0.995996,0.997994,999
9,fra_Latn,0.983034,0.992944,0.987964,992


## Inspect forgetting and recovery

In [4]:
import pandas as pd
for benchmark in results['zero_shot']:
    base = results['zero_shot'][benchmark].set_index('language')
    target = results['target_only'][benchmark].set_index('language')
    replay = results['replay'][benchmark].set_index('language')
    comparison = pd.DataFrame({'zero_shot_f1':base.f1, 'target_only_f1':target.f1,
                               'replay_f1':replay.f1, 'drop_after_target_only':base.f1-target.f1,
                               'improvement_with_replay':replay.f1-target.f1})
    print(benchmark)
    display(comparison)
print('Scores are on a 0–1 scale. Positive drop = forgetting; negative drop = improvement.')

commonlid


,zero_shot_f1,target_only_f1,replay_f1,drop_after_target_only,improvement_with_replay
language,,,,,
sin_Sinh,0.544157,0.968171,0.985203,-0.424014,0.017032
pli_Sinh,0.000000,0.960115,0.990944,-0.960115,0.030829
san_Sinh,0.000000,0.992854,0.992476,-0.992854,-0.000379
san_Deva,0.931442,0.932862,0.930886,-0.001420,-0.001977
eng_Latn,0.923358,0.902631,0.947845,0.020727,0.045214
tam_Taml,0.974684,0.987500,0.987500,-0.012816,0.000000
hin_Deva,0.964136,0.957396,0.960521,0.006740,0.003125
ben_Beng,0.986552,0.986824,0.985734,-0.000273,-0.001091
arb_Arab,0.993111,0.992474,0.992840,0.000637,0.000366


flores_plus


,zero_shot_f1,target_only_f1,replay_f1,drop_after_target_only,improvement_with_replay
language,,,,,
sin_Sinh,0.544157,0.996852,0.985203,-0.452695,-0.011649
pli_Sinh,0.000000,0.996523,0.996358,-0.996523,-0.000165
san_Sinh,0.000000,0.995100,0.994721,-0.995100,-0.000379
san_Deva,0.989011,0.990524,1.000000,-0.001513,0.009476
eng_Latn,0.983957,1.000000,0.966110,-0.016043,-0.033890
tam_Taml,1.000000,1.000000,1.000000,0.000000,0.000000
hin_Deva,0.990682,0.992636,0.999506,-0.001955,0.006869
ben_Beng,1.000000,1.000000,1.000000,0.000000,0.000000
arb_Arab,0.666447,0.666667,0.666667,-0.000220,0.000000


wili_2018


,zero_shot_f1,target_only_f1,replay_f1,drop_after_target_only,improvement_with_replay
language,,,,,
sin_Sinh,0.544157,0.996852,0.985203,-0.452695,-0.011649
pli_Sinh,0.000000,0.996358,0.996358,-0.996358,0.000000
san_Sinh,0.000000,0.990248,0.989869,-0.990248,-0.000379
san_Deva,0.996942,0.996942,0.997967,0.000000,0.001026
eng_Latn,0.930516,0.944762,0.906364,-0.014245,-0.038398
tam_Taml,0.993859,0.993859,0.994373,0.000000,0.000515
hin_Deva,0.989378,0.989378,0.988866,0.000000,-0.000511
ben_Beng,0.942918,0.942918,0.942359,0.000000,-0.000559
arb_Arab,0.997494,0.997994,0.997994,-0.000500,0.000000


Scores are on a 0–1 scale. Positive drop = forgetting; negative drop = improvement.


After all three model notebooks finish, run notebook 04. Do not infer preservation of all original languages from eight replay-language scores. The original label set remains available, but retention needs evaluation.